# V7_D_N03 — See the Service Gaps: Informal-Settlement Mapping

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft. All data are synthetic and illustrative. Analytical outputs require review by the named authority.

## Decision contract
**Decision:** prioritize field verification and equitable service planning. **Owners:** municipal and service authorities. **Horizon:** quarterly to annual. **Boundary:** outputs must not be used for eviction, enforcement, exclusion, or adverse eligibility decisions.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(7303); n=180
grid=pd.DataFrame({'cell_id':[f'G{i:03d}' for i in range(n)],'x_km':rng.uniform(0,20,n),'y_km':rng.uniform(0,15,n),'population':rng.integers(80,950,n),'water_coverage':rng.uniform(.15,1,n),'sanitation_coverage':rng.uniform(.10,1,n),'road_minutes':rng.uniform(5,65,n),'imagery_age_months':rng.integers(1,31,n)})
grid.head().round(2)

## Evidence and rights contract
A remote-sensing or administrative classification is not proof of legal status. Use neutral spatial units, minimize personal data, document uncertainty, and require field/community verification.

In [2]:
assert grid.cell_id.is_unique; assert grid[['water_coverage','sanitation_coverage']].apply(lambda s:s.between(0,1).all()).all(); print('POPULATION',grid.population.sum(),'CELLS',len(grid))

POPULATION 93211 CELLS 180


## Transparent service-gap index
The index combines water, sanitation, and access gaps. Weights are public planning assumptions and require sensitivity analysis.

In [3]:
grid['water_gap']=1-grid.water_coverage; grid['sanitation_gap']=1-grid.sanitation_coverage; grid['access_gap']=(grid.road_minutes/65).clip(0,1); w={'water_gap':.4,'sanitation_gap':.4,'access_gap':.2}; grid['gap_index']=sum(grid[k]*v for k,v in w.items())
grid.gap_index.describe().round(3)

## Population-weighted need
A high gap in an almost empty cell and a moderate gap affecting many residents imply different planning consequences. Report both severity and affected population.

In [4]:
grid['people_equivalent_gap']=grid.population*grid.gap_index; top=grid.nlargest(12,'people_equivalent_gap'); print(top[['cell_id','population','gap_index','people_equivalent_gap']].round(2).to_string(index=False))

cell_id  population  gap_index  people_equivalent_gap
   G054         928       0.79                 734.31
   G065         787       0.81                 637.88
   G036         858       0.66                 570.45
   G067         910       0.62                 568.39
   G021         918       0.61                 562.25
   G015         911       0.61                 559.81
   G167         731       0.75                 544.96
   G084         654       0.79                 517.03
   G116         864       0.60                 516.05
   G079         916       0.55                 505.58
   G113         665       0.76                 502.55
   G050         852       0.58                 497.35


## Data-age abstention
Stale imagery or service records can misdirect resources. Cells with stale evidence are sent to verification instead of being ranked as current fact.

In [5]:
grid['evidence_status']=np.where(grid.imagery_age_months>18,'VERIFY—STALE','CURRENT FOR SCREENING'); eligible=grid[grid.evidence_status=='CURRENT FOR SCREENING'].copy(); print(grid.evidence_status.value_counts().to_string())

evidence_status
CURRENT FOR SCREENING    101
VERIFY—STALE              79


## Spatial access to service points
Distance is calculated here in a synthetic planar coordinate system. Real deployment requires authoritative coordinates, CRS control, network travel time, barriers, and accessibility checks.

In [6]:
services=np.array([[3,3],[10,6],[16,11]]); pts=eligible[['x_km','y_km']].to_numpy(); d=np.sqrt(((pts[:,None,:]-services[None,:,:])**2).sum(axis=2)); eligible['nearest_service_km']=d.min(axis=1); eligible['combined_priority']=.7*eligible.gap_index+.3*(eligible.nearest_service_km/d.max()).clip(0,1)
eligible.nlargest(8,'combined_priority')[['cell_id','gap_index','nearest_service_km','combined_priority']].round(2)

## Geographic equity and capacity
Select a limited field-verification list while ensuring coverage across planning zones. This is not a service-denial rule.

In [7]:
eligible['zone']=pd.cut(eligible.x_km,[0,5,10,15,20],labels=['W','C1','C2','E'],include_lowest=True); worklist=(eligible.sort_values(['zone','combined_priority'],ascending=[True,False]).groupby('zone',observed=True,group_keys=False).head(4)); assert worklist.groupby('zone',observed=True).size().le(4).all(); print(worklist[['cell_id','zone','combined_priority','evidence_status']].round(3).to_string(index=False))

cell_id zone  combined_priority       evidence_status
   G174    W              0.646 CURRENT FOR SCREENING
   G003    W              0.579 CURRENT FOR SCREENING
   G019    W              0.567 CURRENT FOR SCREENING
   G157    W              0.539 CURRENT FOR SCREENING
   G040   C1              0.649 CURRENT FOR SCREENING
   G058   C1              0.571 CURRENT FOR SCREENING
   G165   C1              0.566 CURRENT FOR SCREENING
   G155   C1              0.566 CURRENT FOR SCREENING
   G045   C2              0.664 CURRENT FOR SCREENING
   G036   C2              0.517 CURRENT FOR SCREENING
   G137   C2              0.489 CURRENT FOR SCREENING
   G138   C2              0.488 CURRENT FOR SCREENING
   G084    E              0.645 CURRENT FOR SCREENING
   G144    E              0.598 CURRENT FOR SCREENING
   G096    E              0.553 CURRENT FOR SCREENING
   G081    E              0.506 CURRENT FOR SCREENING


## Weight sensitivity and prohibited-use control
If priorities change greatly under plausible weights, disclose instability. Every output carries an explicit prohibited-use statement.

In [8]:
alt=.25*eligible.water_gap+.55*eligible.sanitation_gap+.20*eligible.access_gap; base=set(eligible.nlargest(20,'gap_index').cell_id); altset=set(eligible.assign(alt=alt).nlargest(20,'alt').cell_id); overlap=len(base&altset)/20; control={'top20_overlap':overlap,'status':'REVIEW' if overlap<.7 else 'STABLE','prohibited_uses':['eviction','enforcement targeting','eligibility denial','public identification of households']}
print(control)

{'top20_overlap': 0.8, 'status': 'STABLE', 'prohibited_uses': ['eviction', 'enforcement targeting', 'eligibility denial', 'public identification of households']}


## Exercises
1. Replace straight-line distance with a network measure. 2. Add uncertainty ranges to coverage estimates. 3. Design a community-validation protocol. 4. Explain why “informal” is not an analytical risk label.

## Exact solutions
1. Use an authoritative routable network, travel modes, barriers, time, accessibility, and CRS validation. 2. Propagate survey/sensor/classification uncertainty and show interval-sensitive ranks. 3. Define representation, consent, accessible communication, correction, grievance, feedback, and no-retaliation safeguards. 4. Informality is a legal/social planning context; converting it into a risk score can stigmatize residents and enable punitive misuse.

In [9]:
assert len(worklist)>0 and set(worklist.evidence_status)=={'CURRENT FOR SCREENING'}
assert 'eviction' in control['prohibited_uses']
print('V7_D_N03_COMPLETE_EXECUTION_PASS')

V7_D_N03_COMPLETE_EXECUTION_PASS
